# Baseline CNN: step by step (original method, unchanged)

This notebook is the notebook version of the two original baseline scripts:

* `extract_faces.py`, which turns each video into face crops;
* `train_cnn.py`, which trains the CNN and evaluates it.

**The method is unchanged.** The functions below are copied from those scripts. Only four technical details differ: three are needed to run inside a Windows notebook, and one lets training use the GPU. None of them changes the method:

| What | Script | Notebook | Why the method stays the same |
|---|---|---|---|
| Participant split | read from the earlier pilot folder | read from `participant_split.csv` in the project | same file: its SHA-256 hash is checked below |
| Parallel face extraction | process pool | thread pool | each video is processed independently by the same function |
| ROC AUC for sex | scikit-learn `roc_auc_score` | the same formula in NumPy | the check in Step 7 reproduces every saved number |
| Computing device | CPU only | the GPU when PyTorch can see one, otherwise the CPU (`DEVICE` in Step 0) | the same calculation on faster hardware; lines marked `added` only move data to and from the GPU. The saved run was trained on the CPU |

**Two switches**, both off by default:

* `RUN_EXTRACTION = False`: reuse the saved face crops in `cache/`. Turning it on needs the raw videos in the pilot folder.
* `RUN_TRAINING = False`: show the **saved run of 5 October 2026** in `results/`. Turning it on trains again (about 1.5 hours on the CPU; faster on the GPU) into a new folder, `results_10_targets/`. The saved run is never overwritten.

**One history note.** The saved 5 October run had 10 regression output slots, one of them an untrained glucose slot without labels. On 10 October the scope changed, and the current code (below) has 9 numerical outputs plus sex. The training method is otherwise identical. The untrained slot never received a loss, so the saved results remain the baseline for the 9 numerical targets and sex.

Use the kernel **`.venv (3.14.6)`**. Graphs are also saved as PNG files in `figures/`.

## Step 0 · Settings

The constants are the same values as in the two scripts.

In [ ]:
# Step 0 · Settings: the constants of extract_faces.py and train_cnn.py (unchanged values)
from __future__ import annotations

import hashlib
import json
import os
import time
from multiprocessing.pool import ThreadPool
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Image, display

RUN_EXTRACTION = False    # True = rebuild cache/faces.npz from the raw videos (needs the pilot's video folder)
RUN_TRAINING = False      # True = train 3 seeds x 30 epochs again into results_10_targets/ (about 1.5 h on CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # added: the GPU when PyTorch can see one
torch.backends.cudnn.deterministic = True                                # added: same seed -> same result on every GPU rerun
torch.backends.cudnn.conv.fp32_precision = 'ieee'                        # added: full 32-bit maths on the GPU, like the CPU


def find_project_root(start):
    for folder in [start, *start.parents]:
        if (folder / 'db.csv').is_file() and (folder / 'baseline_test').is_dir():
            return folder
    raise FileNotFoundError('Open this notebook inside the Face_AI project folder.')


PROJECT = find_project_root(Path.cwd().resolve())
HERE = PROJECT / 'baseline_test'
FIGURES = HERE / 'figures'
FIGURES.mkdir(exist_ok=True)

# --- extract_faces.py constants -------------------------------------------------
RAW_VIDEO_DIR = Path(os.environ.get("MCD_RAW_DIR", PROJECT / "data" / "raw")) / "video"   # raw MCD-rPPG videos
YUNET_ONNX = PROJECT / "experiments" / "improved_cnn" / "weights" / "face_detection_yunet_2026may.onnx"
N_FRAMES = 8          # frames kept per video
WINDOW_S = 60.0       # use the first minute, like the respiratory pilot
CROP = 112            # output crop size in pixels
MARGIN = 1.2          # square box = MARGIN * max(face width, face height); tight to limit background
OVERLAY_ROWS = 60     # the burned-in date/time text occupies the top band; blank it

# --- train_cnn.py constants -----------------------------------------------------
SPLIT_CSV = PROJECT / "participant_split.csv"      # identical copy of the pilot's respiratory-600 split
SPLIT_SHA256 = "f2c8ab6b31f37431c47a2ceaf748466c6c0616ef443a400db848668f48ece94b"   # hash of the pilot's original file
OUT = HERE / "results_10_targets"                  # where a NEW training run would be written
SAVED_RUN = HERE / "results"                       # the saved run of 5 October 2026

# Fixed output order for the scope confirmed on 10 October 2026.
# HbA1c remains its own target, represented once by glycated_hemoglobin.
REGRESSION_TARGETS = ["saturation", "temperature", "glycated_hemoglobin", "cholesterol",
                      "respiratory", "rigidity", "stress", "age", "bmi"]
PRETTY = {"saturation": "Oxygen saturation (%)", "temperature": "Temperature (°C)",
          "glycated_hemoglobin": "HbA1c / glycated haemoglobin (%)",
          "cholesterol": "Cholesterol (mmol/L)", "respiratory": "Respiratory rate (breaths/min)",
          "rigidity": "Arterial stiffness", "stress": "Stress level (PSM-25)", "age": "Age (years)",
          "bmi": "BMI (kg/m²)", "sex": "Sex"}

SEEDS = [0, 1, 2]
EPOCHS = 30
BATCH = 16
LR = 1e-3
WEIGHT_DECAY = 1e-3
BOOTSTRAP = 2000

assert hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest() == SPLIT_SHA256, 'participant_split.csv differs from the original'
print('Project folder:', PROJECT)
print('Split file verified identical to the original pilot split.')
print('Computing on:', f'GPU ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else 'CPU')

# Plot style (shared with the improved notebook): orange = baseline CNN, grey = constant guess
C = dict(cnn='#eb6834', ref='#898781', ink='#0b0b0b', ink2='#52514e', grid='#e1e0d9', axis='#c3c2b7', surface='#fcfcfb')
plt.rcParams.update({
    'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'], 'savefig.facecolor': C['surface'],
    'font.family': 'sans-serif', 'font.sans-serif': ['Segoe UI', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.titlesize': 10.5, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'axes.labelcolor': C['ink2'], 'axes.edgecolor': C['axis'], 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': C['grid'], 'grid.linewidth': 0.8, 'xtick.color': C['axis'], 'ytick.color': C['axis'],
    'xtick.labelcolor': C['ink2'], 'ytick.labelcolor': C['ink2'], 'legend.frameon': False, 'legend.fontsize': 9,
    'lines.linewidth': 1.8, 'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

## Step 1 · Face extraction (from `extract_faces.py`)

For every frontal video, the extraction:

1. reads the first 60 seconds and keeps **8 evenly spaced frames**;
2. blanks the burned-in date/time text at the top;
3. finds the face with the **YuNet** face detector;
4. cuts a square around it (1.2 × the face size);
5. resizes it to **112 × 112 pixels**.

The result is stored once in `cache/faces.npz`.

In [ ]:
# Step 1a · The face-extraction functions (verbatim from extract_faces.py)
def detect_face(detector, frame_bgr):
    """Return (x, y, w, h) of the most confident face in full-resolution pixels, or None."""
    h, w = frame_bgr.shape[:2]
    small = cv2.resize(frame_bgr, (320, 240))
    _, faces = detector.detect(small)
    if faces is None or len(faces) == 0:
        return None
    best = faces[np.argmax(faces[:, -1])]
    sx, sy = w / 320, h / 240
    return best[0] * sx, best[1] * sy, best[2] * sx, best[3] * sy


def square_crop(frame_bgr, box):
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    side = MARGIN * max(w, h)
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + int(round(side)), y0 + int(round(side))
    # pad with black if the square leaves the image
    H, W = frame_bgr.shape[:2]
    pad = max(0, -x0, -y0, x1 - W, y1 - H)
    if pad:
        frame_bgr = cv2.copyMakeBorder(frame_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT)
        x0, y0, x1, y1 = x0 + pad, y0 + pad, x1 + pad, y1 + pad
    crop = frame_bgr[y0:y1, x0:x1]
    crop = cv2.resize(crop, (CROP, CROP), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


def process(stem: str):
    path = RAW_VIDEO_DIR / f"{stem}.avi"
    out = np.zeros((N_FRAMES, CROP, CROP, 3), np.uint8)
    detected = np.zeros(N_FRAMES, bool)
    if not path.exists():
        return stem, out, detected, "missing file"
    detector = cv2.FaceDetectorYN.create(str(YUNET_ONNX), "", (320, 240), 0.8, 0.3, 5000)
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    # The AVI header reports no frame count, so read sequentially and keep chosen indices.
    wanted = np.linspace(0, WINDOW_S * fps - 1, N_FRAMES).round().astype(int)
    wanted_set = {int(i): k for k, i in enumerate(wanted)}
    last_box = None
    idx = -1
    while idx < wanted[-1]:
        ok = cap.grab()
        if not ok:
            break
        idx += 1
        if idx not in wanted_set:
            continue
        ok, frame = cap.retrieve()
        if not ok:
            break
        k = wanted_set[idx]
        frame[:OVERLAY_ROWS] = 0         # hide the timestamp text so it cannot be a shortcut
        box = detect_face(detector, frame)
        if box is not None:
            detected[k] = True
            last_box = box
        if last_box is not None:          # fall back to the previous face box
            out[k] = square_crop(frame, last_box)
    cap.release()
    if idx < wanted[-1]:
        return stem, out, detected, f"video ended at frame {idx}"
    if detected.sum() < N_FRAMES // 2:
        return stem, out, detected, f"face found in only {detected.sum()}/{N_FRAMES} frames"
    # frames before the first detection have no box yet: copy the first good crop back
    first = int(np.argmax(detected))
    out[:first] = out[first]
    return stem, out, detected, "ok"


def extract_all():
    """main() of extract_faces.py; a thread pool replaces the process pool (Windows notebooks cannot fork)."""
    df = pd.read_csv(PROJECT / "db.csv", dtype={"patient_id": str})
    frontal = df[df["camera"] == "FullHDwebcam"]
    stems = sorted(Path(v).stem for v in frontal["video"])
    print(f"{len(stems)} frontal videos to process")
    with ThreadPool(min(20, __import__('os').cpu_count())) as pool:
        results = []
        for i, r in enumerate(pool.imap(process, stems, chunksize=4), 1):
            results.append(r)
            if i % 100 == 0:
                print(f"  {i}/{len(stems)}", flush=True)
    cache = HERE / "cache"
    cache.mkdir(exist_ok=True)
    np.savez_compressed(
        cache / "faces.npz",
        keys=np.array([r[0] for r in results]),
        frames=np.stack([r[1] for r in results]),
        detected=np.stack([r[2] for r in results]),
    )
    log = pd.DataFrame({"video": [r[0] for r in results],
                        "faces_detected": [int(r[2].sum()) for r in results],
                        "status": [r[3] for r in results]})
    log.to_csv(cache / "extraction_log.csv", index=False)
    print(log["status"].eq("ok").value_counts())
    print(log.loc[log["status"] != "ok"].to_string())

In [ ]:
# Step 1b · Run the extraction, or inspect the saved crops
if RUN_EXTRACTION:
    extract_all()
log = pd.read_csv(HERE / 'cache' / 'extraction_log.csv')
print(f"{len(log)} videos in the extraction log; status counts:", log.status.value_counts().to_dict())
print('faces detected per video:', log.faces_detected.value_counts().to_dict())
display(Image(filename=str(HERE / 'cache' / 'qc_contact_sheet.png'), width=900))

**Reading Step 1.** All 1,200 frontal videos produced 8 crops with a detected face. The contact sheet is the quality check made when the cache was built. These same crops are the input of both the baseline and the improved CNN.

## Step 2 · Labels and split (from `train_cnn.py`)

The labels come from `db.csv` (frontal rows only). Every person gets the split from `participant_split.csv`: 420 training, 90 validation and 90 test people.

Two people have a different sex recorded at their two sessions. Their sex label is hidden, not guessed.

In [ ]:
# Step 2 · Data loading (verbatim from train_cnn.py)
def load_table():
    df = pd.read_csv(PROJECT / "db.csv", dtype={"patient_id": str})
    df = df[df["camera"] == "FullHDwebcam"].copy()
    df["key"] = df["video"].map(lambda v: Path(v).stem)
    split = pd.read_csv(SPLIT_CSV, dtype={"patient_id": str})
    df = df.merge(split, on="patient_id", how="left", validate="many_to_one")
    assert df["split"].notna().all(), "every participant must have a split"
    assert df.groupby("patient_id")["split"].nunique().max() == 1, "a person crosses partitions"

    missing = set(REGRESSION_TARGETS) - set(df.columns)
    if missing:
        raise ValueError(f"db.csv is missing required target columns: {sorted(missing)}")
    df["sex_label"] = df["sex"].map({"F": 0.0, "M": 1.0})
    # two people have different sex recorded at the two sessions: do not guess, mask them
    inconsistent = df.groupby("patient_id")["sex"].transform("nunique") > 1
    df.loc[inconsistent, "sex_label"] = np.nan
    return df.reset_index(drop=True), sorted(df.loc[inconsistent, "patient_id"].unique())


def load_frames(keys):
    z = np.load(HERE / "cache" / "faces.npz")
    pos = {k: i for i, k in enumerate(z["keys"])}
    return z["frames"][[pos[k] for k in keys]]          # (n, 8, 112, 112, 3) uint8


df, sex_masked = load_table()
tr, va, te = (df[df["split"] == s].reset_index(drop=True) for s in ("train", "validation", "test"))
print({k: (len(v), v["patient_id"].nunique()) for k, v in {"train": tr, "val": va, "test": te}.items()})
print('sex label hidden for participants:', sex_masked)

## Step 3 · The model (from `train_cnn.py`)

The model has three parts:

* **A small CNN trained from scratch**, with no pretrained weights. It has 4 blocks, each made of two 3 × 3 convolutions with BatchNorm and ReLU, followed by 2 × 2 max-pooling (24 → 48 → 96 → 160 channels).
* **The same CNN runs on each of the 8 crops**, and the 8 results are averaged into one vector per video.
* **Two linear heads**: 9 numerical outputs and 1 sex output (a *logit*, which becomes a probability after a sigmoid).

**Training details:**

* **Augmentation**: horizontal flips and small shifts only, with no colour changes.
* **Loss**: mean squared error on the *standardised* numbers plus binary cross-entropy for sex. Every task has the same weight. Missing labels are skipped (*masked*).

In [ ]:
# Step 3 · Model, augmentation and loss (verbatim from train_cnn.py; predict() also moves data to DEVICE)
def conv_block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout),
                         nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout),
                         nn.ReLU(inplace=True), nn.MaxPool2d(2))


class FaceCNN(nn.Module):
    """Small CNN trained from scratch (no pretrained weights)."""

    def __init__(self, n_reg=len(REGRESSION_TARGETS)):
        super().__init__()
        self.encoder = nn.Sequential(conv_block(3, 24), conv_block(24, 48), conv_block(48, 96),
                                     conv_block(96, 160), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.dropout = nn.Dropout(0.3)
        self.reg_head = nn.Linear(160, n_reg)
        self.sex_head = nn.Linear(160, 1)

    def forward(self, x):                    # x: (batch, frames, 3, H, W) float in [0, 1]
        b, t = x.shape[:2]
        frames = x.flatten(0, 1).contiguous(memory_format=torch.channels_last)  # ~30% faster on CPU
        feats = self.encoder(frames).view(b, t, -1).mean(1)                       # average over frames
        feats = self.dropout(feats)
        return self.reg_head(feats), self.sex_head(feats).squeeze(1)


def to_tensor(frames_uint8):
    """Keep uint8 (n, t, 3, H, W) in memory; convert to float [0, 1] one batch at a time."""
    return torch.from_numpy(frames_uint8).permute(0, 1, 4, 2, 3).contiguous()


def as_float(x_uint8):
    return x_uint8.float().div_(255.0)


def augment(x):
    """Geometric augmentation only: flips and small shifts. No colour jitter, because skin
    colour changes are exactly the signal some targets may depend on."""
    flip = torch.rand(x.shape[0]) < 0.5
    x[flip] = x[flip].flip(-1)
    dx, dy = np.random.randint(-6, 7, size=2)
    return torch.roll(x, shifts=(int(dy), int(dx)), dims=(-2, -1))


def masked_loss(reg_pred, sex_logit, y_reg, y_sex):
    m = ~torch.isnan(y_reg)
    reg = F.mse_loss(reg_pred[m], y_reg[m]) if m.any() else reg_pred.sum() * 0
    ms = ~torch.isnan(y_sex)
    sex = F.binary_cross_entropy_with_logits(sex_logit[ms], y_sex[ms]) if ms.any() else sex_logit.sum() * 0
    # With complete numerical labels, each of the 9 regression tasks has the same
    # weight as sex. If label coverage later differs, use per-task masked means.
    n_reg_tasks = int((~torch.isnan(y_reg)).any(0).sum())
    return (reg * n_reg_tasks + sex) / (n_reg_tasks + 1)


@torch.no_grad()
def predict(model, x, batch=64):
    model.eval()
    regs, sexes = [], []
    for i in range(0, len(x), batch):
        r, s = model(as_float(x[i:i + batch]).to(DEVICE))            # .to(DEVICE) added
        regs.append(r.cpu()); sexes.append(torch.sigmoid(s).cpu())    # .cpu() added
    return torch.cat(regs).numpy(), torch.cat(sexes).numpy()


# Layer table: output shape for one 112x112 crop and parameters per block (inspection only)
model = FaceCNN().eval()
rows, x = [], torch.zeros(1, 3, 112, 112)
with torch.no_grad():
    for i, block in enumerate(model.encoder):
        x = block(x)
        rows.append({'block': f'encoder[{i}] {type(block).__name__}', 'output shape (one crop)': ' × '.join(map(str, x.shape[1:])),
                     'parameters': sum(p.numel() for p in block.parameters())})
rows.append({'block': 'average over the 8 crops + dropout 30%', 'output shape (one crop)': '160 per video', 'parameters': 0})
rows.append({'block': 'reg_head (9 numbers) + sex_head (1 logit)', 'output shape (one crop)': '10 per video',
             'parameters': sum(p.numel() for p in model.reg_head.parameters()) + sum(p.numel() for p in model.sex_head.parameters())})
display(pd.DataFrame(rows).set_index('block'))
print(f'total parameters (current 9-output code): {sum(p.numel() for p in model.parameters()):,}; '
      'the saved 5 October run had one extra (unlabelled) output slot: 533,075')
del model

## Step 4 · Training (from `train_cnn.py`)

Each seed trains for 30 epochs with these settings:

* **Optimiser**: AdamW, learning rate 0.001, weight decay 0.001.
* **Schedule**: cosine learning-rate decay.
* **Checkpoint**: after every epoch the model is measured on the validation people, and the epoch with the **lowest validation loss** is kept.
* **Ensemble**: the 3 seeds are averaged.

In [ ]:
# Step 4 · One training run per seed (verbatim from train_cnn.py; only the DEVICE moves marked "added" are new)
def train_one_seed(seed, x_tr, yr_tr, ys_tr, x_va, yr_va, ys_va):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FaceCNN().to(DEVICE, memory_format=torch.channels_last)                # DEVICE added
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    best, best_state, history = np.inf, None, []
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); losses = []
        for idx in torch.randperm(len(x_tr)).split(BATCH):
            r, s = model(augment(as_float(x_tr[idx])).to(DEVICE))                  # .to(DEVICE) added
            loss = masked_loss(r, s, yr_tr[idx].to(DEVICE), ys_tr[idx].to(DEVICE))   # .to(DEVICE) added
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        sched.step()
        model.eval()
        with torch.no_grad():
            r, s = [], []
            for i in range(0, len(x_va), 64):
                a, b = model(as_float(x_va[i:i + 64]).to(DEVICE)); r.append(a.cpu()); s.append(b.cpu())   # moves added
            val_loss = masked_loss(torch.cat(r), torch.cat(s), yr_va, ys_va).item()
        history.append({"seed": seed, "epoch": epoch, "train_loss": float(np.mean(losses)),
                        "val_loss": val_loss, "seconds": time.time() - t0})
        print(f"seed {seed} epoch {epoch:2d} train {np.mean(losses):.4f} val {val_loss:.4f} "
              f"({time.time() - t0:.0f}s)", flush=True)
        if val_loss < best:
            best = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history

## Step 5 · Evaluation functions (from `train_cnn.py`)

* **MAE per person.** Errors are averaged within each person first, then across people, so every person counts once.
* **Skill** = 1 − CNN MAE / constant MAE. The constant is the training median, or the majority class for sex.
* **95% interval.** A paired bootstrap over people gives the interval of (CNN MAE − constant MAE). A negative interval favours the CNN.

The only change: `roc_auc_score` is written out in NumPy with the same definition, because the project kernel has no scikit-learn.

In [ ]:
# Step 5 · Evaluation (verbatim from train_cnn.py, except roc_auc_score is defined here with NumPy)
def roc_auc_score(y_true, y_score):
    """Same value as sklearn.metrics.roc_auc_score for binary labels (Mann-Whitney rank formula, ties averaged)."""
    y_true, ranks = np.asarray(y_true), pd.Series(np.asarray(y_score)).rank().to_numpy()
    n1 = (y_true == 1).sum(); n0 = len(y_true) - n1
    return (ranks[y_true == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def person_mae(abs_err, pids):
    """Average errors within each person first, then across people (one vote per person)."""
    return pd.Series(abs_err).groupby(np.asarray(pids)).mean().mean()


def paired_bootstrap(err_model, err_base, pids, rng):
    """95% interval of (model MAE - baseline MAE), resampling people with replacement."""
    per = pd.DataFrame({"m": err_model, "b": err_base, "p": pids}).groupby("p").mean()
    d = (per["m"] - per["b"]).to_numpy()
    boots = rng.choice(d, size=(BOOTSTRAP, len(d)), replace=True).mean(1)
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def evaluate(part, frame, reg_pred, sex_prob, baselines, rng):
    rows = []
    pids = frame["patient_id"].to_numpy()
    for j, t in enumerate(REGRESSION_TARGETS):
        y = frame[t].to_numpy(dtype=float)
        ok = ~np.isnan(y)
        if not ok.any():
            rows.append({"partition": part, "target": t, "type": "regression", "n_people": 0,
                         "note": "no reference labels in db.csv"})
            continue
        e_m = np.abs(reg_pred[ok, j] - y[ok]); e_b = np.abs(baselines[t] - y[ok])
        mae_m, mae_b = person_mae(e_m, pids[ok]), person_mae(e_b, pids[ok])
        lo, hi = paired_bootstrap(e_m, e_b, pids[ok], rng)
        rows.append({"partition": part, "target": t, "type": "regression",
                     "n_people": int(pd.Series(pids[ok]).nunique()),
                     "cnn_mae": mae_m, "baseline_mae": mae_b,
                     "skill_vs_baseline": 1 - mae_m / mae_b if mae_b > 0 else np.nan,
                     "mae_diff_ci_low": lo, "mae_diff_ci_high": hi,
                     "pearson_r": float(np.corrcoef(reg_pred[ok, j], y[ok])[0, 1]),
                     "pred_std": float(reg_pred[ok, j].std()), "label_std": float(y[ok].std())})
    y = frame["sex_label"].to_numpy(dtype=float); ok = ~np.isnan(y)
    pred = (sex_prob[ok] >= 0.5).astype(float)
    base = np.full(ok.sum(), baselines["sex"])
    def bal_acc(p, yy):
        return np.mean([np.mean(p[yy == c] == c) for c in (0, 1)])
    rows.append({"partition": part, "target": "sex", "type": "classification",
                 "n_people": int(pd.Series(pids[ok]).nunique()),
                 "cnn_accuracy": float(np.mean(pred == y[ok])),
                 "baseline_accuracy": float(np.mean(base == y[ok])),
                 "cnn_balanced_accuracy": float(bal_acc(pred, y[ok])),
                 "baseline_balanced_accuracy": float(bal_acc(base, y[ok])),
                 "cnn_auc": float(roc_auc_score(y[ok], sex_prob[ok]))})
    return rows

## Step 6 · Run the experiment, or load the saved run

With `RUN_TRAINING = True`, the cell below executes `main()` of `train_cnn.py`, step for step: training-only constant baselines and standardisation, 3 seeds, ensemble predictions, evaluation and saved files. By default it loads the saved run of 5 October instead.

In [ ]:
# Step 6 · Train (main() of train_cnn.py) or use the saved 5 October run
baselines = {t: float(tr[t].median()) for t in REGRESSION_TARGETS if tr[t].notna().any()}
baselines["sex"] = float(tr.drop_duplicates("patient_id")["sex_label"].dropna().mode()[0])

if RUN_TRAINING:
    # Preserve the archived run in results/ and never overwrite a previous new run.
    OUT.mkdir(exist_ok=False)
    mu = tr[REGRESSION_TARGETS].mean().to_numpy(); sd = tr[REGRESSION_TARGETS].std().to_numpy()
    mu = np.nan_to_num(mu); sd = np.where(np.isfinite(sd) & (sd > 0), sd, 1.0)

    def tensors(frame):
        x = to_tensor(load_frames(frame["key"].tolist()))
        yr = torch.tensor((frame[REGRESSION_TARGETS].to_numpy(float) - mu) / sd, dtype=torch.float32)
        ys = torch.tensor(frame["sex_label"].to_numpy(float), dtype=torch.float32)
        return x, yr, ys

    (x_tr, yr_tr, ys_tr), (x_va, yr_va, ys_va), (x_te, _, _) = map(tensors, (tr, va, te))
    histories, preds = [], {"validation": [], "test": []}
    for seed in SEEDS:
        model, hist = train_one_seed(seed, x_tr, yr_tr, ys_tr, x_va, yr_va, ys_va)
        histories += hist
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, OUT / f"cnn_seed{seed}.pt")   # .cpu() added
        for part, x in (("validation", x_va), ("test", x_te)):
            r, s = predict(model, x)
            preds[part].append((r * sd + mu, s))           # back to original units
    n_params = sum(p.numel() for p in FaceCNN().parameters())
    pd.DataFrame(histories).to_csv(OUT / "training_history.csv", index=False)
    rng = np.random.default_rng(20261005)
    rows = []
    for part, frame in (("validation", va), ("test", te)):
        reg = np.mean([p[0] for p in preds[part]], 0)
        sex = np.mean([p[1] for p in preds[part]], 0)
        rows += evaluate(part, frame, reg, sex, baselines, rng)
        out = frame[["patient_id", "step", "key"]].copy()
        for j, t in enumerate(REGRESSION_TARGETS):
            out[f"pred_{t}"] = reg[:, j]; out[f"true_{t}"] = frame[t]
        out["pred_prob_male"] = sex; out["true_sex"] = frame["sex"]
        out.to_csv(OUT / f"{part}_predictions.csv", index=False)
    res = pd.DataFrame(rows)
    res.to_csv(OUT / "metrics.csv", index=False)
    reg = res[res["type"] == "regression"].dropna(subset=["cnn_mae"])
    v = reg[reg.partition == "validation"].set_index("target")
    t = reg[reg.partition == "test"].set_index("target")
    ranking = pd.DataFrame({"validation_skill": v["skill_vs_baseline"],
                            "test_skill": t["skill_vs_baseline"],
                            "test_cnn_mae": t["cnn_mae"], "test_baseline_mae": t["baseline_mae"],
                            "test_diff_ci": [f"[{a:+.3f}, {b:+.3f}]" for a, b in
                                             zip(t["mae_diff_ci_low"], t["mae_diff_ci_high"])],
                            "test_pearson_r": t["pearson_r"]}).sort_values("validation_skill", ascending=False)
    ranking.index = ranking.index.map(PRETTY)
    ranking.to_csv(OUT / "ranking.csv")
    json.dump({"scope_version": "10-targets-2026-10-10",
               "regression_targets": REGRESSION_TARGETS, "classification_target": "sex",
               "n_parameters": n_params, "seeds": SEEDS, "epochs": EPOCHS, "device": str(DEVICE),   # device added
               "best_epochs": pd.DataFrame(histories).loc[lambda h: h.groupby("seed")["val_loss"].idxmin(), ["seed", "epoch", "val_loss"]].to_dict("records"),
               "baselines_fitted_on_train": baselines, "sex_label_masked_participants": sex_masked,
               "people": {"train": int(tr.patient_id.nunique()), "validation": int(va.patient_id.nunique()),
                          "test": int(te.patient_id.nunique())}},
              open(OUT / "run_info.json", "w"), indent=2)
    RUN = OUT
else:
    RUN = SAVED_RUN
info = json.loads((RUN / 'run_info.json').read_text(encoding='utf-8'))
history = pd.read_csv(RUN / 'training_history.csv')
metrics = pd.read_csv(RUN / 'metrics.csv')
print('Showing the run in:', RUN.relative_to(PROJECT))
print('kept epochs (lowest validation loss):', info['best_epochs'])
print(f"logged training time: {history.seconds.sum() / 60:.0f} minutes for {history.seed.nunique()} seeds x {history.epoch.max()} epochs")

## Step 7 · Check: the notebook code reproduces the saved numbers

To prove that this notebook evaluates exactly like the original script, we feed the saved predictions into `evaluate()` with the same random seed. Every saved metric must come out identical, including the bootstrap intervals and the sex AUC computed with NumPy.

In [ ]:
# Step 7 · Recompute every saved metric from the saved predictions with this notebook's evaluate()
rng = np.random.default_rng(20261005)
recomputed = []
for part, frame in (('validation', va), ('test', te)):
    saved = pd.read_csv(RUN / f'{part}_predictions.csv', dtype={'patient_id': str}).set_index('key').loc[frame.key]
    reg = saved[[f'pred_{t}' for t in REGRESSION_TARGETS]].to_numpy(float)
    recomputed += evaluate(part, frame, reg, saved['pred_prob_male'].to_numpy(float), baselines, rng)
recomputed = pd.DataFrame(recomputed).set_index(['partition', 'target'])
saved_metrics = metrics.set_index(['partition', 'target']).loc[recomputed.index]
columns = [c for c in ['cnn_mae', 'baseline_mae', 'skill_vs_baseline', 'mae_diff_ci_low', 'mae_diff_ci_high', 'pearson_r',
                       'pred_std', 'label_std', 'cnn_accuracy', 'baseline_accuracy', 'cnn_balanced_accuracy',
                       'baseline_balanced_accuracy', 'cnn_auc'] if c in recomputed]
a, b = recomputed[columns].to_numpy(float), saved_metrics[columns].to_numpy(float)
same = np.isclose(a, b, rtol=1e-9, atol=1e-12) | (np.isnan(a) & np.isnan(b))
assert same.all(), recomputed[columns][~same.all(axis=1)]
print(f'OK: all {same.size} saved metric values ({len(recomputed)} rows x {len(columns)} columns) are reproduced exactly.')

## Step 8 · Graphs of the baseline run

In [ ]:
# Step 8a · Training curves: 3 seeds x 30 epochs, kept epoch = lowest validation loss
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.9), sharey=True, layout='constrained')
for ax, seed in zip(axes, sorted(history.seed.unique())):
    h = history[history.seed == seed].sort_values('epoch')
    best = next(b for b in info['best_epochs'] if b['seed'] == seed)
    ax.axvline(best['epoch'], color=C['ref'], lw=1)
    ax.text(best['epoch'] - 0.6, 1.092, f"kept: epoch {best['epoch']}", ha='right', va='top', fontsize=8.5, color=C['ink2'])
    ax.plot(h.epoch, h.train_loss, color=C['cnn'], alpha=0.45)
    ax.plot(h.epoch, h.val_loss, color=C['cnn'])
    ax.scatter([best['epoch']], [best['val_loss']], s=46, color=C['cnn'], edgecolor=C['surface'], linewidth=1.4, zorder=3)
    ax.text(h.epoch.iloc[-1] + 0.6, h.train_loss.iloc[-1], 'training', fontsize=8.5, color=C['ink2'], va='center')
    ax.text(h.epoch.iloc[-1] + 0.6, h.val_loss.iloc[-1], 'validation', fontsize=8.5, color=C['ink2'], va='center')
    if h.val_loss.max() > 1.1:
        ax.text(0.03, 0.03, f'axis cut at 1.1: early validation\nspikes reach {h.val_loss.max():.2f}',
                transform=ax.transAxes, fontsize=8, color=C['ink2'], va='bottom')
    ax.set_xlim(0, 37)
    ax.set_ylim(0.7, 1.1)
    ax.set_title(f'Seed {seed}')
    ax.set_xlabel('Epoch')
axes[0].set_ylabel('Combined loss (lower = better)')
fig.suptitle('Baseline CNN · training curves: validation loss bottoms out only at epochs 24–29 of 30',
             x=0.01, ha='left', fontweight='semibold', fontsize=12)
fig.savefig(FIGURES / 'baseline_loss_curves.png')
plt.show()

In [ ]:
# Step 8b · Skill vs the constant guess per target, validation and test (with the test 95% interval)
reg_metrics = metrics[(metrics.type == 'regression') & metrics.cnn_mae.notna()].set_index(['partition', 'target'])
order = reg_metrics.loc['validation'].skill_vs_baseline.sort_values().index.tolist()     # ranking fixed on validation
fig, ax = plt.subplots(figsize=(9.5, 5), layout='constrained')
yy = np.arange(len(order))
test = reg_metrics.loc['test'].loc[order]
low, high = -test.mae_diff_ci_high / test.baseline_mae * 100, -test.mae_diff_ci_low / test.baseline_mae * 100
ax.hlines(yy, low, high, color=C['cnn'], lw=2)
ax.scatter(100 * reg_metrics.loc['validation'].loc[order].skill_vs_baseline, yy, s=46, facecolor=C['surface'],
           edgecolor=C['cnn'], linewidth=1.6, zorder=3, label='validation')
ax.scatter(100 * test.skill_vs_baseline, yy, s=46, color=C['cnn'], edgecolor=C['surface'], linewidth=1.4, zorder=3,
           label='test (line = 95% interval)')
ax.axvline(0, color=C['ink2'], lw=1)
ax.set_yticks(yy, [PRETTY[t] for t in order])
ax.set_xlabel('Error reduction vs constant guess (%)  ·  left of 0 = worse than always guessing the training median')
ax.legend(loc='lower right')
fig.suptitle('Baseline CNN · only age and BMI clearly beat a constant guess', x=0.01, ha='left', fontweight='semibold', fontsize=12)
fig.savefig(FIGURES / 'baseline_skill.png')
plt.show()
display(pd.read_csv(RUN / 'ranking.csv', index_col=0).round(3))
display(metrics[metrics.type == 'classification'].dropna(axis=1).set_index('partition').round(3))

In [ ]:
# Step 8c · Predicted vs true on the test people
saved = pd.read_csv(RUN / 'test_predictions.csv', dtype={'patient_id': str})
fig, axes = plt.subplots(3, 3, figsize=(11, 11), layout='constrained')
for ax, t in zip(axes.flat, REGRESSION_TARGETS):
    truth, pred = saved[f'true_{t}'].to_numpy(float), saved[f'pred_{t}'].to_numpy(float)
    values = np.r_[truth, pred]
    pad = (values.max() - values.min()) * 0.05
    lim = (values.min() - pad, values.max() + pad)
    ax.plot(lim, lim, color=C['ref'], lw=1)
    ax.scatter(truth, pred, s=14, color=C['cnn'], alpha=0.6, edgecolor='none')
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect('equal')
    ax.set_title(PRETTY[t])
    ax.text(0.97, 0.03, f'spread kept {np.std(pred) / np.std(truth):.0%}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8.5, color=C['ink2'])
for ax in axes[-1]:
    ax.set_xlabel('True value')
for ax in axes[:, 0]:
    ax.set_ylabel('Predicted value')
fig.suptitle('Baseline CNN · predicted vs true (test): physiological predictions stay near the median',
             x=0.01, ha='left', fontweight='semibold', fontsize=12)
fig.savefig(FIGURES / 'baseline_predicted_vs_true.png')
plt.show()

**Reading the baseline.**

* **The from-scratch CNN learns appearance.** Sex is near-perfect, age correlates strongly (r ≈ 0.92 on test) and BMI improves modestly.
* **It does not beat the constant guess on the physiological targets.** Their predictions stay close to the median.
* **The validation loss was still lowest near the end of the 30 epochs** (epochs 24–29), so the model was under-trained rather than over-fitted.

The improved pipeline in `experiments/improved_cnn/improved_cnn.ipynb` keeps the same crops and the same split, and changes the model step by step.